In [67]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

model = "claude-haiku-4-5"

In [68]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [69]:
import json  

def generate_dataset():

    prompt = """

Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts

that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,

each representing task that requires Python, JSON, or a Regex to complete.

  

Example output:

```json

[

	{

		"task": "Description of task",
        "format": "python/json/regex"        
		
	},
		
	...additional

]

``` * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.

* Focus on tasks that do not require writing much code

Please generate 3 objects.

"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  # 프리필링
    text = chat(messages, stop_sequences=["```"])  # 정지 시퀀스
    return json.loads(text)

In [70]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [71]:
def run_prompt(test_case):
    """프롬프트와 테스트 케이스를 병합하여 실행"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation

"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")  # 프리필링
    output = chat(messages, stop_sequences=["```"])  # 정지 시퀀스
    return output

In [72]:
test_case = dataset[0]  # 첫 번째 테스트 케이스 선택
print(test_case)

run_prompt(test_case)

{'task': 'Write a Python function that validates if a given string is a valid AWS S3 bucket name (3-63 characters, lowercase letters, numbers, and hyphens only, cannot start or end with hyphen)', 'format': 'python'}


'\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    """\n    Validates if a given string is a valid AWS S3 bucket name.\n    \n    Rules:\n    - 3-63 characters long\n    - Lowercase letters, numbers, and hyphens only\n    - Cannot start or end with hyphen\n    """\n    pattern = r\'^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$\'\n    return bool(re.match(pattern, bucket_name))\n\n\n# Test cases\ntest_cases = [\n    ("my-bucket", True),\n    ("mybucket", True),\n    ("my-bucket-123", True),\n    ("a", False),\n    ("ab", False),\n    ("abc", True),\n    ("-mybucket", False),\n    ("mybucket-", False),\n    ("my_bucket", False),\n    ("my.bucket", False),\n    ("MyBucket", False),\n    ("my-bucket" * 10, False),\n    ("a" * 63, True),\n    ("a" * 64, False),\n    ("", False),\n    ("my--bucket", True),\n    ("123bucket", True),\n]\n\nfor bucket_name, expected in test_cases:\n    result = is_valid_s3_bucket_name(bucket_name)\n    status = "✓" if result == expected else "✗"

In [73]:
def grade_by_model(test_case, output):
    """LLM-as-Judge: 다른 Claude 호출로 출력을 평가"""
    eval_prompt = f"""

You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.  

Original Task:
<task>
{test_case["task"]}
</task> 

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10  

Respond with JSON. Keep your response concise and direct.

Example response shape:

{{
"strengths": string[],
"weaknesses": string[],
"reasoning": string,
"score": number
}}

"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])	
    return json.loads(eval_text)

In [74]:
import re
import ast  

def validate_json(text):
	try:
		json.loads(text.strip())
		return 10
	except json.JSONDecodeError:
		return 0
		
def validate_python(text):
	try:
		ast.parse(text.strip())
		return 10
	except SyntaxError:
		return 0 

def validate_regex(text):
	try:
		re.compile(text.strip())
		return 10
	except re.error:
		return 0

def grade_syntax(response, test_case):
	format = test_case["format"]
	if format == "json":
		return validate_json(response)
	elif format == "python":
		return validate_python(response)
	else:
		return validate_regex(response)

In [75]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    # 모델 채점
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    # 최종 점수는 모델 평가 점수와 문법 유효성 점수의 평균
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [76]:
def run_eval(dataset):
    """데이터셋의 모든 테스트 케이스를 순차 실행"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = sum(r["score"] for r in results) / len(results)
    print(f"Average Score: {average_score:.2f}")

    return results

In [77]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average Score: 8.50
[
  {
    "output": "\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    pattern = r'^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$'\n    return bool(re.match(pattern, bucket_name))\n",
    "test_case": {
      "task": "Write a Python function that validates if a given string is a valid AWS S3 bucket name (3-63 characters, lowercase letters, numbers, and hyphens only, cannot start or end with hyphen)",
      "format": "python"
    },
    "score": 9.0,
    "reasoning": "The solution correctly implements the core S3 bucket naming rules through a well-constructed regex pattern. The pattern `^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$` validates length (3-63 chars), character set, and hyphen placement accurately. However, the function lacks defensive programming by not validating input type before regex matching, which is a practical concern in production code. The optional group correctly handles the edge case of 1-2 character inputs being invalid."
  },
  {
    "output": "\n

In [78]:
print(results)

[{'output': "\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    pattern = r'^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$'\n    return bool(re.match(pattern, bucket_name))\n", 'test_case': {'task': 'Write a Python function that validates if a given string is a valid AWS S3 bucket name (3-63 characters, lowercase letters, numbers, and hyphens only, cannot start or end with hyphen)', 'format': 'python'}, 'score': 9.0, 'reasoning': 'The solution correctly implements the core S3 bucket naming rules through a well-constructed regex pattern. The pattern `^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$` validates length (3-63 chars), character set, and hyphen placement accurately. However, the function lacks defensive programming by not validating input type before regex matching, which is a practical concern in production code. The optional group correctly handles the edge case of 1-2 character inputs being invalid.'}, {'output': '\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Effec